In [ ]:
import os
from google.cloud import bigquery

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "/home/user/GITs/arxiv-data-pipeline/credentials/pipeline-sa.json"
)
os.environ["GCP_PROJECT_ID"] = "arxiv-data-pipeline"

GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
BQ_DATASET = os.getenv("BQ_DATASET", "arxiv_dataset")

bq_client = bigquery.Client(project=GCP_PROJECT_ID)
print(f"Connected to project: {GCP_PROJECT_ID}")

In [30]:
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

In [31]:
AutoTokenizer.from_pretrained("allenai/specter2_base")
m = AutoAdapterModel.from_pretrained("allenai/specter2_base")
m.load_adapter("allenai/specter2", source="hf", load_as="proximity")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'proximity'

## BigQuery Connection

In [32]:
# Inspect all relevant table schemas
for table_id in [
    "arxiv_dataset.fct_papers",
    "arxiv_dataset.fct_papers_embeddings",
    "arxiv_dataset.paper_embeddings",
    "arxiv_dataset.paper_keywords",
    "ingestion_dataset.papers",
]:
    t = bq_client.get_table(f"{GCP_PROJECT_ID}.{table_id}")
    print(f"\n{'=' * 60}")
    print(f"{table_id}  ({t.num_rows:,} rows)")
    print(f"{'=' * 60}")
    for field in t.schema:
        mode = f"[{field.mode}]" if field.mode != "NULLABLE" else ""
        print(f"  {field.name:<30} {field.field_type:<12} {mode}")


arxiv_dataset.fct_papers  (347,221 rows)
  arxiv_id                       STRING       
  title                          STRING       
  authors                        STRING       [REPEATED]
  primary_category               STRING       
  all_categories                 STRING       [REPEATED]
  category                       STRING       
  subject_group                  STRING       
  version                        INTEGER      
  doi                            STRING       
  journal_ref                    STRING       
  is_published                   BOOLEAN      
  comment                        STRING       
  date_published                 TIMESTAMP    
  date_updated                   DATE         
  year                           INTEGER      
  month                          INTEGER      
  has_code                       BOOLEAN      
  is_survey                      BOOLEAN      

arxiv_dataset.fct_papers_embeddings  (347,221 rows)
  arxiv_id                       STRING

In [34]:
# Sanity check: count papers and embeddings
df = bq_client.query(f"""
    SELECT
        (SELECT COUNT(*) FROM `{GCP_PROJECT_ID}.{BQ_DATASET}.fct_papers`) AS total_papers,
        (SELECT COUNT(*) FROM `{GCP_PROJECT_ID}.{BQ_DATASET}.paper_embeddings`) AS total_embedded
""").to_dataframe(create_bqstorage_client=False)

print(df.to_string(index=False))

 total_papers  total_embedded
       347221          347221


In [35]:
# Peek at a few embeddings rows
df_sample = bq_client.query(f"""
    SELECT 
        arxiv_id, 
        keywords, 
        ARRAY_LENGTH(embedding) AS embedding_dim, 
        embedded_at
    FROM 
        `{GCP_PROJECT_ID}.{BQ_DATASET}.paper_embeddings`
    LIMIT 5
""").to_dataframe(create_bqstorage_client=False)

df_sample

,arxiv_id,keywords,embedding_dim,embedded_at
0,2502.11504,"[bilevel optimization, nature-inspired optimiz...",768,2026-04-24 17:13:18.846775+00:00
1,2502.10390,"[sketch, offline rl, self-knowledge distillati...",768,2026-04-24 17:13:18.846775+00:00
2,2305.16196,"[network embedding, graph attention, learning ...",768,2026-04-24 17:13:18.846775+00:00
3,2305.16074,"[multi-armed bandits, long-tail learning, offl...",768,2026-04-24 17:13:18.846775+00:00
4,2305.18732,"[long-tail learning with class descriptors, lw...",768,2026-04-24 17:13:18.846775+00:00


## VECTOR_SEARCH Test

In [36]:
import torch
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

MODEL_ID = "allenai/specter2_base"
ADAPTER_ID = "allenai/specter2"

device = torch.device("cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoAdapterModel.from_pretrained(MODEL_ID)
model.load_adapter(ADAPTER_ID, source="hf", load_as="proximity", set_active=True)
model = model.to(device)
model.eval()
print("Model loaded.")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Model loaded.


In [47]:
@torch.no_grad()
def embed_query(text: str) -> list[float]:
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=512, padding=True
    )
    outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().float().tolist()


# Test query - change this to anything you want to search for
QUERY = "diffusion models for image generationw"

query_embedding = embed_query(QUERY)
print(f"Embedding dim: {len(query_embedding)}")

Embedding dim: 768


In [48]:
TOP_K = 10
embedding_str = ", ".join(str(x) for x in query_embedding)

results = bq_client.query(f"""
    SELECT
        base.arxiv_id,
        base.title,
        ARRAY_TO_STRING(base.authors, ', ') AS authors,
        base.date_published,
        base.primary_category,
        p.abstract,
        ROUND(1 - q.distance, 4) AS similarity
    FROM VECTOR_SEARCH(
        TABLE `{GCP_PROJECT_ID}.{BQ_DATASET}.paper_embeddings`,
        'embedding',
        (SELECT [{embedding_str}] AS embedding),
        top_k => {TOP_K},
        distance_type => 'COSINE'
    ) q
    JOIN `{GCP_PROJECT_ID}.{BQ_DATASET}.fct_papers` base
        ON base.arxiv_id = q.base.arxiv_id
    JOIN `{GCP_PROJECT_ID}.ingestion_dataset.papers` p
        ON p.arxiv_id = q.base.arxiv_id
    ORDER BY q.distance
""").to_dataframe(create_bqstorage_client=False)

results[["arxiv_id", "title", "primary_category", "date_published", "similarity"]]

,arxiv_id,title,primary_category,date_published,similarity
0,2310.03431,Robust Zero Level-Set Extraction from Unsigned...,cs.CV,2023-10-05 00:00:00+00:00,0.9106
1,2407.03172,IMC 2024 Methods & Solutions Review,cs.CV,2024-07-03 00:00:00+00:00,0.9081
2,2012.10660,The importance of silhouette optimization in 3...,cs.CV,2020-12-19 00:00:00+00:00,0.9078
3,2412.06257,Advancing Extended Reality with 3D Gaussian Sp...,cs.CV,2024-12-09 00:00:00+00:00,0.9075
4,2103.08737,Growing 3D Artefacts and Functional Machines w...,cs.LG,2021-03-15 00:00:00+00:00,0.9067
5,2310.14707,A Hybrid GNN approach for predicting node data...,cs.LG,2023-10-23 00:00:00+00:00,0.9064
6,2407.17418,"3D Gaussian Splatting: Survey, Technologies, C...",cs.CV,2024-07-24 00:00:00+00:00,0.9064
7,2405.03417,Gaussian Splatting: 3D Reconstruction and Nove...,cs.CV,2024-05-06 00:00:00+00:00,0.9060
8,2011.08771,A Method to Generate High Precision Mesh Model...,cs.CV,2020-11-17 00:00:00+00:00,0.9060
9,1906.07870,Analytical Derivatives for Differentiable Rend...,cs.CV,2019-06-19 00:00:00+00:00,0.9055


In [ ]:
top = results.iloc[0]
print(f"Title:      {top['title']}")
print(f"ID:         {top['arxiv_id']}")
print(f"Category:   {top['primary_category']}")
print(f"Published:  {top['date_published']}")
print(f"Similarity: {top['similarity']}")
print(f"Authors:    {top['authors']}")
print(f"\nAbstract:\n{top['abstract']}")